In [37]:
# Quoridor Game

In [38]:
from dataclasses import dataclass, replace
from collections import deque
import random

In [39]:
N = 5 # board width and length
WN = N - 1 # wall slots form a WN x WN grid
WALLS_PER_PLAYER = 3

NORTH = (-1, 0)
SOUTH = (1, 0)
EAST = (0, 1)
WEST = (0, -1)
DIRECTIONS = (NORTH, SOUTH, EAST, WEST)

GOAL_ROW = (N - 1, 0) #player 0 aims for the bottom row, player 1 for the top

def slot_bit(r, c):
    # bit flat for wall slot (r, c)
    return 1 << (r * WN + c)

@dataclass(frozen=True)
class State:
    pawns: tuple #((r0, c0), (r1, c1)) positions for player 1 and 2
    walls_left: tuple # (walls_left_for_p0, walls_left_for_p1)
    h_walls: int # bitmask of occupied horizontal slots
    v_walls: int # bitmask of occupied vertical slots
    turn: int # 0 or 1, whose move it is

def initial_state():
    mid = N//2

    return State(
        pawns=((0, mid), (N - 1, mid)), # start the players in the middle column
        walls_left=(WALLS_PER_PLAYER, WALLS_PER_PLAYER),
        h_walls=0,
        v_walls=0,
        turn=0,
    )


In [40]:
def in_bounds(r, c):
    return 0 <=r < N and 0 <= c < N

def is_blocked(state, r, c, direction):
    dr, dc = direction
    if dr != 0:
        # step in the verical direction, the groove lies bewteen rows, so pick the slot row
        groove_row = r if dr == 1 else r - 1
        mask = state.h_walls
        slots = ((groove_row, c), (groove_row, c - 1))

    else:
        groove_col = c if dc == 1 else c - 1
        mask = state.v_walls
        slots = ((r, groove_col), (r - 1, groove_col))

    for sr, sc in slots:
        if 0 <=sr < WN and 0 <= sc < WN and mask & slot_bit(sr, sc):
            return True

    return False
        

In [41]:
s = initial_state()
s = State(s.pawns, s.walls_left, s.h_walls | slot_bit(1, 1), s.v_walls, s.turn)

print(is_blocked(s, 1, 1, SOUTH))   # True
print(is_blocked(s, 1, 2, SOUTH))   # True, same wall covers column 2
print(is_blocked(s, 1, 3, SOUTH))   # False
print(is_blocked(s, 2, 1, NORTH))   # True, blocking is symmetric
print(is_blocked(s, 1, 1, EAST))    # False, horizontal walls don't block sideways

True
True
False
True
False


In [42]:
def pawn_moves(state):
    # squares the player to move can step to, including jumps over the opponent
    me = state.pawns[state.turn]
    them = state.pawns[1-state.turn]
    moves = []

    for direction in DIRECTIONS:
        dr, dc = direction
        r, c = me[0] + dr, me[1] + dc

        if not in_bounds(r, c) or is_blocked(state, me[0], me[1], direction):
            continue
        if (r, c) != them:
            moves.append((r,c))
            continue

        # opponent is in the target square so we try jumping over
        jr, jc = r + dr, c + dc
        if in_bounds(jr, jc) and not is_blocked(state, r, c, direction):
            moves.append((jr, jc))
            continue

        # blocked behind them, so sidestep to either square perpendituclar to the player
        for side in ((dc, dr), (-dc, -dr)):
            sr, sc = r + side[0], c + side[1]
            if in_bounds(sr, sc) and not is_blocked(state, r, c, side):
                moves.append((sr, sc))

    return moves

In [43]:
def path_exists(state, player):
    # BFS can this player still reach their goal row?
    goal = GOAL_ROW[player]
    start = state.pawns[player]
    seen = {start}
    q = deque([start])

    while q:
        r, c = q.popleft()
        if r == goal:
            return True
        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            if (in_bounds(nr, nc) 
            and (nr, nc) not in seen 
            and not is_blocked(state, r, c, (dr, dc))
            ):
                seen.add((nr, nc))
                q.append((nr, nc))

    return False

In [44]:
def h_conflict(state, r, c):
    """
    a horizontal wall at (r, c) covers columns c and c+1 so a horizontal
    neighbour one column either side would overlap half of it
    """
    for cc in (c - 1, c + 1):
        if 0 <= cc < WN and state.h_walls & slot_bit(r, cc):
            return True

    return False

def v_conflict(state, r, c):
    for rr in (r - 1, r + 1):
        if 0 <=rr < WN and state.v_walls & slot_bit(rr, c):
            return True
    return False

def wall_moves(state):
    if state.walls_left[state.turn] == 0:
        return []

    moves = []
    for r in range(WN):
        for c in range(WN):
            bit = slot_bit(r, c)


            # nothing of either oreintation may already sit on this intersection
            if (state.h_walls | state.v_walls) & bit:
                continue
                
            if not h_conflict(state, r, c):
                cand = replace(state, h_walls=state.h_walls | bit)
                if path_exists(cand, 0) and path_exists(cand, 1):
                    moves.append(('H', r, c))

            if not v_conflict(state, r, c):
                cand = replace(state, v_walls=state.v_walls | bit)
                if path_exists(cand, 0) and path_exists(cand, 1):
                    moves.append(('V', r, c))

    return moves

In [45]:
def legal_moves(state):
    return [('P', r, c) for r, c in pawn_moves(state)] + wall_moves(state)

def apply_move(state, move):
    # the successor function, every agent's compute cost is counted in calls to this
    kind, r, c = move
    nxt = 1 - state.turn

    if kind == 'P':
        pawns = list(state.pawns)
        pawns[state.turn] = (r, c)
        return replace(state, pawns=tuple(pawns), turn=nxt)

    walls_left = list(state.walls_left)
    walls_left[state.turn] -= 1
    bit = slot_bit(r, c)

    if kind == 'H':
        return replace(state, walls_left=tuple(walls_left), h_walls=state.h_walls | bit, turn=nxt)
    return replace(state, walls_left=tuple(walls_left), v_walls=state.v_walls | bit, turn=nxt)

def winner(state):
    # 0 or 1 if that player reached their goal row, otherwise None
    for player in (0, 1):
        if state.pawns[player][0] == GOAL_ROW[player]:
            return player
    return None

In [48]:
def random_move(state, rng):
    # one uniformly random legal move, without validating the walls we dont play
    candidates = [('P', r, c) for r, c in pawn_moves(state)]

    if state.walls_left[state.turn] > 0:
        for r in range(WN):
            for c in range(WN):
                bit = slot_bit(r, c)
                if (state.h_walls | state.v_walls) & bit:
                    continue
                if not h_conflict(state, r, c):
                    candidates.append(('H', r, c))
                if not v_conflict(state, r, c):
                    candidates.append(('V', r, c))

    while candidates:
        move = rng.choice(candidates)
        if move[0] == 'P':
            return move
        cand = apply_move(state, move)
        if path_exists(cand, 0) and path_exists(cand, 1):
            return move
        candidates.remove(move) # that wall cuts someone off, never offer it again

    return None

In [49]:
def random_playout(rng, max_plies=300):
    state = initial_state()
    for _ in range(max_plies):
        w = winner(state)
        if w is not None:
            return w

        state = apply_move(state, random_move(state, rng))
    return None

# s = initial_state()
# rng = random.Random(0)
# res = [random_playout(rng) for _ in range(1000)]

# print("opening moves:", len(legal_moves(s)), "=", len(pawn_moves(s)), "pawn +", len(wall_moves(s)), "wall")
# print(f"P0 {res.count(0)}  P1 {res.count(1)}  unfinished {res.count(None)}")

opening moves: 35 = 3 pawn + 32 wall
P0 444  P1 536  unfinished 20
